# Portfolio Growth Engine — Analysis Notebook

Interactive analysis of your investment portfolio.
Run cells top-to-bottom. Edit `data/portfolio.yaml` with your real holdings first.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.portfolio import load_portfolio
from src.market_data import update_portfolio_prices
from src.analyzer import (
    holding_performance_table,
    asset_class_allocation,
    sector_allocation,
    concentration_risk,
    calculate_portfolio_cagr,
    calculate_portfolio_xirr,
)
from src.goal_tracker import track_all_goals, growth_scenarios, milestone_table
from src.allocator import calculate_rebalance, get_current_allocation, AGGRESSIVE_GROWTH

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Portfolio

In [ ]:
portfolio = load_portfolio()

print(f'Holdings: {len(portfolio.holdings)}')
print(f'Goals: {len(portfolio.goals)}')
print(f'Total Invested: ₹{portfolio.total_invested:,.0f}')
print(f'Current Value:  ₹{portfolio.total_current_value:,.0f}')
print(f'P&L:            ₹{portfolio.total_pnl:,.0f} ({portfolio.total_pnl_percent:+.1f}%)')
print(f'CAGR:           {calculate_portfolio_cagr(portfolio):.1f}%')

## 2. Holdings Performance

In [ ]:
df = holding_performance_table(portfolio)
df

## 3. Asset Allocation — Pie Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# By asset class
ac = asset_class_allocation(portfolio)
labels = [v['label'] for v in ac.values()]
sizes = [v['value'] for v in ac.values()]
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
axes[0].set_title('By Asset Class')

# By sector
sec = sector_allocation(portfolio)
labels_s = list(sec.keys())
sizes_s = [v['value'] for v in sec.values()]
axes[1].pie(sizes_s, labels=labels_s, autopct='%1.1f%%', startangle=90)
axes[1].set_title('By Sector')

plt.tight_layout()
plt.show()

## 4. Concentration Risk

In [ ]:
conc = concentration_risk(portfolio)
print(f"Top 5 holdings = {conc['top_percent']}% of portfolio")
print()
for h in conc['top_holdings']:
    print(f"  {h['symbol']:15s}  ₹{h['value']:>10,.0f}  ({h['percent']}%)")

## 5. Goal Progress — 1000x Tracker

In [ ]:
for gp in track_all_goals(portfolio):
    status = '✅ ON TRACK' if gp.on_track else '❌ BEHIND'
    print(f'\n=== {gp.goal.name} === {status}')
    print(f'  Target:       {gp.goal.target_multiplier:.0f}x in {gp.goal.target_years} years')
    print(f'  Current:      ₹{gp.current_value:,.0f}')
    print(f'  Target Value: ₹{gp.target_value:,.0f}')
    print(f'  Required CAGR:      {gp.required_cagr}%')
    print(f'  Required from now:  {gp.required_cagr_from_now}%')
    print(f'  Actual CAGR:        {gp.actual_cagr}%')
    print(f'  Completion:         {gp.completion_percent}% (log scale)')

## 6. Growth Scenarios — 20 Year Projection

In [ ]:
initial = portfolio.total_invested
years = np.arange(0, 21)
rates = [12, 15, 20, 25, 30, 40, 50]

plt.figure(figsize=(14, 7))
for r in rates:
    values = initial * (1 + r/100) ** years
    plt.plot(years, values / 1e7, label=f'{r}% CAGR', linewidth=2)

plt.xlabel('Years')
plt.ylabel('Portfolio Value (₹ Crores)')
plt.title(f'Growth Projections — Starting ₹{initial:,.0f}')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

# Table
scenarios = growth_scenarios(initial)
pd.DataFrame(scenarios)

## 7. Rebalancing Suggestions

In [ ]:
actions = calculate_rebalance(portfolio)
rebal_df = pd.DataFrame([
    {
        'Bucket': a.bucket,
        'Current %': a.current_percent,
        'Target %': a.target_percent,
        'Diff %': a.diff_percent,
        'Action': a.action,
        'Amount (₹)': a.amount,
    }
    for a in actions
])
rebal_df

In [ ]:
# Visual: Current vs Target allocation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

current = get_current_allocation(portfolio)
target = AGGRESSIVE_GROWTH

all_buckets = sorted(set(list(current.keys()) + list(target.keys())))

x = np.arange(len(all_buckets))
width = 0.35

cur_vals = [current.get(b, 0) for b in all_buckets]
tgt_vals = [target.get(b, 0) for b in all_buckets]

axes[0].barh(x - width/2, cur_vals, width, label='Current', color='#4CAF50')
axes[0].barh(x + width/2, tgt_vals, width, label='Target', color='#2196F3')
axes[0].set_yticks(x)
axes[0].set_yticklabels(all_buckets)
axes[0].set_xlabel('Allocation %')
axes[0].set_title('Current vs Target Allocation')
axes[0].legend()

# Diff
diffs = [target.get(b, 0) - current.get(b, 0) for b in all_buckets]
colors = ['green' if d > 0 else 'red' for d in diffs]
axes[1].barh(all_buckets, diffs, color=colors)
axes[1].set_xlabel('Difference (Target - Current) %')
axes[1].set_title('Rebalancing Gap')
axes[1].axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

## 8. Stock Research — Screen for Multi-baggers

In [ ]:
from src.screener import screen_multiple, NIFTY_50_POPULAR, SMALL_CAP_WATCHLIST

# Screen a few stocks (uncomment to run — takes ~30s per batch)
# results = screen_multiple(['RELIANCE', 'TCS', 'HDFCBANK', 'DIXON', 'KAYNES'])
# pd.DataFrame([vars(r) for r in results]).sort_values('score', ascending=False)

---
## Next Steps

1. **Replace sample data** in `data/portfolio.yaml` with your real holdings
2. **Run with live prices**: `python -m src.cli summary --live`
3. **Screen stocks**: `python -m src.cli screen --theme Defence`
4. **Track goals monthly** — update prices, re-run analysis
5. **Add new holdings** as you invest — track every transaction